# 0050 v2-B3 High Risk Sample Review

這份 Notebook 用來研究 v2-B3 剩餘高風險樣本。它不是 v2-C，不是主策略升級，而是 Future Work / Execution Research。

In [ ]:
# -*- coding: utf-8 -*-
"""
0050_v2B3_high_risk_sample_review
目的：針對 v2-B3 final signal candidate 的剩餘高風險樣本做分層、標籤、成本敏感與 proxy scenario review。

建議輸入：
1) 最推薦：0050_v2B3_final_stratified_review_outputs.xlsx
   - 內含 Parsed_B3_Events sheet，已經有完整事件欄位。
2) 可替代：TradingView v2-B3 raw trade list CSV
   - 程式會自動解析 ENTRY / EXIT 訊號字串。

輸出：
0050_v2B3_high_risk_sample_review_outputs.xlsx

重要提醒：
這份研究不是 v2-C，不是主策略版本升級。
它是 execution / high-risk review，用來判斷剩餘弱點屬於訊號問題、成本問題、出場問題、重複訊號衰減，還是市場狀態問題。
"""

import os
import re
import glob
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ============================================================
# 0. 使用者參數
# ============================================================
TICK_SIZE = 0.05
SLIPPAGE_TICK_EACH_SIDE = 0.25
SLIPPAGE_ROUND_TRIP_COST = TICK_SIZE * SLIPPAGE_TICK_EACH_SIDE * 2  # 0.025
FIXED_COST_001 = 0.01
FIXED_COST_002 = 0.02

OUTPUT_FILE = "0050_v2B3_high_risk_sample_review_outputs.xlsx"

# ============================================================
# 1. 讀檔工具
# ============================================================

def read_csv_flexible(path):
    """自動嘗試常見編碼讀取 CSV。"""
    encodings = ["utf-8-sig", "utf-8", "cp950", "big5"]
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    raise last_err


def parse_signal_features(signal):
    """解析 TradingView 訊號字串，例如 ENTRY|RD=-0.24|T5=0.26。"""
    out = {}
    if pd.isna(signal):
        return out
    parts = str(signal).split("|")
    if len(parts) > 0:
        out["signal_type_raw"] = parts[0]
    for p in parts[1:]:
        if "=" not in p:
            continue
        k, v = p.split("=", 1)
        k = k.strip()
        v = v.strip()
        # 嘗試轉成數字；失敗保留文字
        try:
            if v == "":
                out[k] = np.nan
            else:
                out[k] = float(v)
        except Exception:
            out[k] = v
    return out


def parse_tradingview_raw_csv(path):
    """把 TradingView raw trade list 解析成一筆交易一列的 event dataframe。"""
    raw = read_csv_flexible(path)
    required_cols = ["Trade number", "類型", "日期和時間", "訊號", "價格 TWD", "淨損益 TWD"]
    missing = [c for c in required_cols if c not in raw.columns]
    if missing:
        raise ValueError(f"這不是可解析的 TradingView raw CSV，缺少欄位：{missing}")

    raw["日期和時間"] = pd.to_datetime(raw["日期和時間"], errors="coerce")
    raw["價格 TWD"] = pd.to_numeric(raw["價格 TWD"], errors="coerce")
    raw["淨損益 TWD"] = pd.to_numeric(raw["淨損益 TWD"], errors="coerce")

    if "有利波動 TWD" in raw.columns:
        raw["有利波動 TWD"] = pd.to_numeric(raw["有利波動 TWD"], errors="coerce")
    if "不利波動 TWD" in raw.columns:
        raw["不利波動 TWD"] = pd.to_numeric(raw["不利波動 TWD"], errors="coerce")

    events = []
    for trade_no, g in raw.groupby("Trade number"):
        entry_rows = g[g["類型"].astype(str).str.contains("進場", na=False)]
        exit_rows = g[g["類型"].astype(str).str.contains("出場", na=False)]
        if entry_rows.empty or exit_rows.empty:
            continue

        entry = entry_rows.sort_values("日期和時間").iloc[0]
        exit_ = exit_rows.sort_values("日期和時間").iloc[-1]

        entry_feat = parse_signal_features(entry["訊號"])
        exit_feat = parse_signal_features(exit_["訊號"])

        entry_dt = entry["日期和時間"]
        exit_dt = exit_["日期和時間"]
        base_pnl = exit_["淨損益 TWD"]

        row = {
            "version": "v2-B3_final_candidate",
            "trade_number": trade_no,
            "entry_datetime": entry_dt,
            "exit_datetime": exit_dt,
            "entry_date": entry_dt.date() if pd.notna(entry_dt) else pd.NaT,
            "entry_year": entry_dt.year if pd.notna(entry_dt) else np.nan,
            "entry_month": entry_dt.strftime("%Y-%m") if pd.notna(entry_dt) else np.nan,
            "entry_time_hhmm": entry_dt.strftime("%H:%M") if pd.notna(entry_dt) else np.nan,
            "exit_time_hhmm": exit_dt.strftime("%H:%M") if pd.notna(exit_dt) else np.nan,
            "entry_price": entry["價格 TWD"],
            "exit_price": exit_["價格 TWD"],
            "base_pnl_twd": base_pnl,
            "mfe_twd": exit_.get("有利波動 TWD", np.nan),
            "mae_twd": exit_.get("不利波動 TWD", np.nan),
            "entry_signal": entry["訊號"],
            "exit_signal": exit_["訊號"],
            "exit_reason": str(exit_["訊號"]).split("|")[0] if pd.notna(exit_["訊號"]) else np.nan,
        }

        # ENTRY features
        mapping = {
            "MODE": "mode",
            "RD": "entry_relative_deviation",
            "VR": "volume_ratio_at_entry",
            "S1": "deviation_slope_1bar",
            "S3": "deviation_slope_3bar",
            "E5": "etf_return_5m",
            "E15": "etf_return_15m",
            "T5": "tw50_return_5m",
            "T15": "tw50_return_15m",
            "RV10": "realized_vol_10bar",
            "MSO": "minutes_since_open",
            "MTC": "minutes_to_close",
            "SID": "session_id",
            "SCT": "signal_count_today",
            "T5OK": "t5_filter_ok",
            "B1OK": "b1_filter_ok",
            "B2OK": "b2_filter_ok",
            "B3OK": "b3_filter_ok",
        }
        for src, dst in mapping.items():
            row[dst] = entry_feat.get(src, np.nan)

        # EXIT features
        row["exit_relative_deviation"] = exit_feat.get("XRD", np.nan)
        row["bars_held"] = exit_feat.get("BH", np.nan)
        if pd.notna(entry_dt) and pd.notna(exit_dt):
            row["holding_minutes"] = (exit_dt - entry_dt).total_seconds() / 60
        else:
            row["holding_minutes"] = np.nan

        events.append(row)

    df = pd.DataFrame(events)
    return add_derived_columns(df)


def add_derived_columns(df):
    """補上成本、時段、資料切分與基本衍生欄位。"""
    out = df.copy()
    out["entry_datetime"] = pd.to_datetime(out["entry_datetime"], errors="coerce")
    out["exit_datetime"] = pd.to_datetime(out["exit_datetime"], errors="coerce")
    out["entry_date"] = pd.to_datetime(out["entry_date"], errors="coerce")

    if "entry_year" not in out.columns or out["entry_year"].isna().all():
        out["entry_year"] = out["entry_datetime"].dt.year
    if "entry_month" not in out.columns or out["entry_month"].isna().all():
        out["entry_month"] = out["entry_datetime"].dt.strftime("%Y-%m")
    if "entry_time_hhmm" not in out.columns or out["entry_time_hhmm"].isna().all():
        out["entry_time_hhmm"] = out["entry_datetime"].dt.strftime("%H:%M")

    numeric_cols = [
        "base_pnl_twd", "entry_relative_deviation", "volume_ratio_at_entry",
        "deviation_slope_1bar", "deviation_slope_3bar",
        "etf_return_5m", "etf_return_15m", "tw50_return_5m", "tw50_return_15m",
        "realized_vol_10bar", "minutes_since_open", "minutes_to_close",
        "signal_count_today", "bars_held", "holding_minutes", "mfe_twd", "mae_twd"
    ]
    for c in numeric_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    # 成本情境
    out["pnl_no_cost"] = out["base_pnl_twd"]
    out["pnl_cost_001"] = out["base_pnl_twd"] - FIXED_COST_001
    out["pnl_cost_002"] = out["base_pnl_twd"] - FIXED_COST_002
    out["pnl_slip_025tick"] = out["base_pnl_twd"] - SLIPPAGE_ROUND_TRIP_COST

    # 資料切分
    def split_year(y):
        if pd.isna(y): return "Unknown"
        if int(y) <= 2024: return "Train_2022_2024"
        if int(y) == 2025: return "Validation_2025"
        return "Recent_2026"
    out["data_split"] = out["entry_year"].apply(split_year)

    # 時段 bucket：刻意把 exact 09:45 放到 09:45-10:00，方便檢查邊界。
    def session_bucket(hhmm):
        if pd.isna(hhmm): return "Unknown"
        t = str(hhmm)[:5]
        if "09:05" <= t < "09:45": return "早盤_0905_0945"
        if "09:45" <= t < "10:00": return "早盤中段_0945_1000"
        if "10:00" <= t < "12:45": return "盤中_1000_1245"
        if "12:45" <= t <= "13:00": return "午盤尾段_1245_1300"
        return "Other"
    if "session_block" not in out.columns or out["session_block"].isna().all():
        out["session_block"] = out["entry_time_hhmm"].apply(session_bucket)

    out["is_exact_0945"] = out["entry_time_hhmm"].astype(str).str[:5].eq("09:45")
    out["is_exact_1000"] = out["entry_time_hhmm"].astype(str).str[:5].eq("10:00")
    out["is_lunch_tail"] = out["session_block"].eq("午盤尾段_1245_1300")
    out["is_repeated_signal"] = out["signal_count_today"].fillna(1) >= 2
    out["is_exit_time"] = out["exit_reason"].astype(str).eq("EXIT_TIME")

    # 高波動 top 20% 標記
    rv_q80 = out["realized_vol_10bar"].quantile(0.8) if "realized_vol_10bar" in out.columns else np.nan
    out["rv10_q80_threshold"] = rv_q80
    out["is_high_rv_top20"] = out["realized_vol_10bar"] >= rv_q80 if pd.notna(rv_q80) else False

    # holding bucket
    def holding_bucket(x):
        if pd.isna(x): return "Unknown"
        if x <= 5: return "<=5m"
        if x <= 10: return "5-10m"
        if x <= 20: return "10-20m"
        return ">20m"
    out["holding_bucket"] = out["holding_minutes"].apply(holding_bucket)

    return out


def load_input():
    """Colab 上傳或本地自動找檔。"""
    if IN_COLAB:
        print("請上傳 0050_v2B3_final_stratified_review_outputs.xlsx 或 v2-B3 raw TradingView CSV")
        uploaded = files.upload()
        path = list(uploaded.keys())[0]
    else:
        candidates = glob.glob("*v2B3*final*review*outputs*.xlsx") + glob.glob("*v2-B3*.csv") + glob.glob("*v2B3*.csv")
        if not candidates:
            raise FileNotFoundError("找不到 v2-B3 final review xlsx 或 raw csv。")
        path = candidates[0]
    print("Using input:", path)

    if path.lower().endswith(".xlsx"):
        xls = pd.ExcelFile(path)
        print("Sheets:", xls.sheet_names)
        if "Parsed_B3_Events" not in xls.sheet_names:
            raise ValueError("Excel 中找不到 Parsed_B3_Events。請改上傳 final stratified review outputs 或 raw CSV。")
        df = pd.read_excel(path, sheet_name="Parsed_B3_Events")
        df = add_derived_columns(df)
    elif path.lower().endswith(".csv"):
        df = parse_tradingview_raw_csv(path)
    else:
        raise ValueError("只支援 .xlsx 或 .csv")

    print("Loaded events:", df.shape)
    display(df.head(3))
    return df

# ============================================================
# 2. 統計函數
# ============================================================

def profit_factor(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gp = pnl[pnl > 0].sum()
    gl = pnl[pnl < 0].sum()
    if gl < 0:
        return gp / abs(gl)
    if gp > 0 and gl == 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0)
    cum = pnl.cumsum()
    dd = cum.cummax() - cum
    return dd.max()


def perf_summary(df, pnl_col="pnl_no_cost", group_cols=None):
    def _calc(g):
        pnl = pd.to_numeric(g[pnl_col], errors="coerce").dropna()
        trades = len(pnl)
        if trades == 0:
            return pd.Series({
                "trades": 0, "wins": 0, "losses": 0, "flats": 0,
                "win_rate": np.nan, "loss_rate": np.nan, "flat_rate": np.nan,
                "net_pnl": np.nan, "avg_pnl": np.nan, "median_pnl": np.nan,
                "gross_profit": np.nan, "gross_loss": np.nan, "profit_factor": np.nan,
                "max_single_loss": np.nan, "max_single_profit": np.nan,
                "estimated_max_drawdown": np.nan
            })
        return pd.Series({
            "trades": trades,
            "wins": int((pnl > 0).sum()),
            "losses": int((pnl < 0).sum()),
            "flats": int((pnl == 0).sum()),
            "win_rate": float((pnl > 0).mean()),
            "loss_rate": float((pnl < 0).mean()),
            "flat_rate": float((pnl == 0).mean()),
            "net_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "gross_profit": float(pnl[pnl > 0].sum()),
            "gross_loss": float(pnl[pnl < 0].sum()),
            "profit_factor": profit_factor(pnl),
            "max_single_loss": float(pnl.min()),
            "max_single_profit": float(pnl.max()),
            "estimated_max_drawdown": max_drawdown_from_pnl(pnl),
        })

    if group_cols is None:
        return _calc(df).to_frame().T
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    return df.groupby(group_cols, dropna=False).apply(_calc).reset_index()


def add_pretty_round(df):
    out = df.copy()
    for c in out.columns:
        if c not in ["trades", "wins", "losses", "flats"]:
            if pd.api.types.is_numeric_dtype(out[c]):
                out[c] = out[c].round(4)
    return out

# ============================================================
# 3. 高風險標籤
# ============================================================

def build_risk_tags(row):
    tags = []

    # 直接虧損
    if row.get("pnl_no_cost", np.nan) < 0:
        tags.append("Loss_Trade")

    # 成本敏感：原始不虧，但扣成本後不再有正 edge
    if row.get("pnl_no_cost", np.nan) > 0 and row.get("pnl_cost_001", np.nan) <= 0:
        tags.append("Cost_Fragile_001")
    if row.get("pnl_no_cost", np.nan) > 0 and row.get("pnl_cost_002", np.nan) <= 0:
        tags.append("Cost_Fragile_002")
    if row.get("pnl_no_cost", np.nan) > 0 and row.get("pnl_slip_025tick", np.nan) <= 0:
        tags.append("Slip_Fragile_025tick")

    # 出場問題
    if row.get("is_exit_time", False):
        tags.append("EXIT_TIME_No_Reversion")

    # 同日重複訊號
    if row.get("is_repeated_signal", False):
        tags.append("Repeated_Signal")

    # 午盤尾段
    if row.get("is_lunch_tail", False):
        tags.append("Lunch_Tail_1245_1300")

    # 高短線波動
    if row.get("is_high_rv_top20", False):
        tags.append("High_RV10_Top20pct")

    # Train 早期樣本
    if row.get("data_split") == "Train_2022_2024":
        tags.append("Train_2022_2024")

    # 09:45 邊界
    if row.get("is_exact_0945", False):
        tags.append("Boundary_Exact_0945")

    if not tags:
        tags.append("Normal_or_Unclassified")
    return "|".join(tags)


def make_tag_long(df):
    tag_long = df[[
        "trade_number", "entry_datetime", "entry_year", "entry_month", "data_split",
        "session_block", "exit_reason", "signal_count_today", "base_pnl_twd",
        "pnl_no_cost", "pnl_cost_001", "pnl_cost_002", "pnl_slip_025tick",
        "entry_relative_deviation", "tw50_return_5m", "realized_vol_10bar",
        "risk_tags"
    ]].copy()
    tag_long["risk_tag"] = tag_long["risk_tags"].str.split("|")
    tag_long = tag_long.explode("risk_tag")
    return tag_long


def tag_summary(tag_long, pnl_cols):
    rows = []
    for pnl_col in pnl_cols:
        temp = perf_summary(tag_long, pnl_col=pnl_col, group_cols="risk_tag")
        temp.insert(0, "pnl_col", pnl_col)
        rows.append(temp)
    return pd.concat(rows, ignore_index=True)

# ============================================================
# 4. Scenario proxy tests
# ============================================================

def build_scenarios(df):
    """建立 future work proxy。這些不是主策略定案，只是研究哪類弱點值得後續處理。"""
    d = df.copy()

    # 門檻：更深偏離 = entry_relative_deviation 低於 Q25；更強 TW50 = T5 高於 Q75
    rd_q25 = d["entry_relative_deviation"].quantile(0.25)
    t5_q75 = d["tw50_return_5m"].quantile(0.75)
    rv_q80 = d["realized_vol_10bar"].quantile(0.80)

    scenarios = {}
    scenarios["Base_v2B3"] = d

    # 午盤尾段研究：不是直接砍，而是觀察更高門檻是否改善成本後品質
    scenarios["LunchTail_keep_deeper_RD_only"] = d[(~d["is_lunch_tail"]) | (d["entry_relative_deviation"] <= rd_q25)]
    scenarios["LunchTail_keep_stronger_TW50_only"] = d[(~d["is_lunch_tail"]) | (d["tw50_return_5m"] >= t5_q75)]
    scenarios["LunchTail_keep_deepRD_or_strongT5"] = d[(~d["is_lunch_tail"]) | (d["entry_relative_deviation"] <= rd_q25) | (d["tw50_return_5m"] >= t5_q75)]
    scenarios["Exclude_LunchTail_diagnostic_only"] = d[~d["is_lunch_tail"]]

    # 重複訊號研究：第 2 筆以後要求更高 expected edge
    scenarios["Repeated_require_deeper_RD"] = d[(~d["is_repeated_signal"]) | (d["entry_relative_deviation"] <= rd_q25)]
    scenarios["Repeated_require_stronger_TW50"] = d[(~d["is_repeated_signal"]) | (d["tw50_return_5m"] >= t5_q75)]
    scenarios["Repeated_require_deepRD_or_strongT5"] = d[(~d["is_repeated_signal"]) | (d["entry_relative_deviation"] <= rd_q25) | (d["tw50_return_5m"] >= t5_q75)]

    # 高波動研究：確認高波動是否真的該避開
    scenarios["Exclude_HighRV_top20_diagnostic_only"] = d[~d["is_high_rv_top20"]]
    scenarios["Exclude_HighRV_and_EXIT_TIME"] = d[~(d["is_high_rv_top20"] & d["is_exit_time"])]

    # EXIT_TIME 研究：僅作診斷，不建議直接把 EXIT_TIME 當進場前可知條件
    scenarios["Exclude_EXIT_TIME_diagnostic_only_not_tradable"] = d[~d["is_exit_time"]]

    # 嚴格版本：只作 future work，不作主策略
    scenarios["Strict_execution_proxy"] = d[
        (~d["is_lunch_tail"]) &
        (~(d["is_repeated_signal"] & (d["entry_relative_deviation"] > rd_q25) & (d["tw50_return_5m"] < t5_q75))) &
        (~(d["is_high_rv_top20"] & d["is_exit_time"]))
    ]

    meta = pd.DataFrame({
        "threshold_name": ["rd_q25_deeper_is_lower", "tw50_t5_q75_stronger_is_higher", "rv10_q80_high_vol"],
        "value": [rd_q25, t5_q75, rv_q80],
        "meaning": [
            "entry_relative_deviation <= this means deeper discount / stronger lag",
            "tw50_return_5m >= this means stronger TW50 short-term upward move",
            "realized_vol_10bar >= this means top 20% short-term vol"
        ]
    })

    return scenarios, meta


def scenario_summary(scenarios, pnl_cols):
    rows = []
    base_n = len(scenarios["Base_v2B3"])
    for name, d in scenarios.items():
        for pnl_col in pnl_cols:
            s = perf_summary(d, pnl_col=pnl_col)
            s.insert(0, "pnl_col", pnl_col)
            s.insert(0, "scenario", name)
            s.insert(2, "kept_trades", len(d))
            s.insert(3, "removed_trades", base_n - len(d))
            s.insert(4, "kept_ratio", len(d) / base_n if base_n else np.nan)
            rows.append(s)
    return pd.concat(rows, ignore_index=True)

# ============================================================
# 5. 主流程
# ============================================================

df = load_input()

# 基本檢查
print("Events count:", len(df))
if len(df) != 223:
    print("⚠️ 注意：目前事件數不是 223。若你是跑完整 v2-B3，理論上應為 223 筆。請確認是否讀到完整檔案。")

# 加高風險標籤
review = df.copy()
review["risk_tags"] = review.apply(build_risk_tags, axis=1)
tag_long = make_tag_long(review)

PNL_COLS = ["pnl_no_cost", "pnl_cost_001", "pnl_cost_002", "pnl_slip_025tick"]

# 主要績效表
overall = pd.concat([perf_summary(review, pnl_col=c).assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
by_split = pd.concat([perf_summary(review, pnl_col=c, group_cols="data_split").assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
by_session = pd.concat([perf_summary(review, pnl_col=c, group_cols="session_block").assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
by_exit = pd.concat([perf_summary(review, pnl_col=c, group_cols="exit_reason").assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
by_signal_count = pd.concat([perf_summary(review, pnl_col=c, group_cols="signal_count_today").assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
by_rv = pd.concat([perf_summary(review, pnl_col=c, group_cols="is_high_rv_top20").assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)

# 高風險標籤表
tag_perf = tag_summary(tag_long, PNL_COLS)
tag_x_split = pd.concat([perf_summary(tag_long, pnl_col=c, group_cols=["risk_tag", "data_split"]).assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
tag_x_session = pd.concat([perf_summary(tag_long, pnl_col=c, group_cols=["risk_tag", "session_block"]).assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
tag_x_exit = pd.concat([perf_summary(tag_long, pnl_col=c, group_cols=["risk_tag", "exit_reason"]).assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)

# 高風險交互：幾個關鍵組合
session_x_exit = pd.concat([perf_summary(review, pnl_col=c, group_cols=["session_block", "exit_reason"]).assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
repeated_x_session = pd.concat([perf_summary(review, pnl_col=c, group_cols=["is_repeated_signal", "session_block"]).assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)
highrv_x_exit = pd.concat([perf_summary(review, pnl_col=c, group_cols=["is_high_rv_top20", "exit_reason"]).assign(pnl_col=c) for c in PNL_COLS], ignore_index=True)

# Scenario proxy
scenarios, thresholds = build_scenarios(review)
scenario_perf = scenario_summary(scenarios, PNL_COLS)
scenario_by_split = []
for name, d in scenarios.items():
    for c in PNL_COLS:
        temp = perf_summary(d, pnl_col=c, group_cols="data_split")
        temp.insert(0, "scenario", name)
        temp.insert(1, "pnl_col", c)
        scenario_by_split.append(temp)
scenario_by_split = pd.concat(scenario_by_split, ignore_index=True)

# Worst casebook
case_cols = [
    "trade_number", "entry_datetime", "exit_datetime", "entry_year", "entry_month", "data_split",
    "entry_time_hhmm", "session_block", "exit_reason", "signal_count_today",
    "entry_price", "exit_price", "base_pnl_twd", "pnl_cost_001", "pnl_cost_002", "pnl_slip_025tick",
    "holding_minutes", "holding_bucket", "mfe_twd", "mae_twd",
    "entry_relative_deviation", "deviation_slope_1bar", "deviation_slope_3bar",
    "tw50_return_5m", "tw50_return_15m", "etf_return_5m", "etf_return_15m",
    "volume_ratio_at_entry", "realized_vol_10bar", "risk_tags", "entry_signal", "exit_signal"
]
case_cols = [c for c in case_cols if c in review.columns]
worst_loss_base = review.sort_values("pnl_no_cost", ascending=True)[case_cols].head(30)
worst_loss_cost002 = review.sort_values("pnl_cost_002", ascending=True)[case_cols].head(30)
cost_fragile_cases = review[(review["pnl_no_cost"] > 0) & (review["pnl_cost_002"] <= 0)][case_cols].sort_values("base_pnl_twd", ascending=True)

# 研究結論文字
conclusion_text = f"""
0050 v2-B3 High Risk Sample Review 結論摘要：

1. 本研究不是建立 v2-C，也不是立即新增主策略濾網，而是針對 v2-B3 final signal candidate 的剩餘高風險樣本進行風險分類。
2. 高風險樣本分為 Loss_Trade、Cost_Fragile、EXIT_TIME_No_Reversion、Repeated_Signal、Lunch_Tail_1245_1300、High_RV10_Top20pct、Train_2022_2024 等類型。
3. 研究重點不是看到哪一類虧損就直接砍掉，而是判斷其本質屬於 signal risk、execution risk、exit risk、regime risk，或 normal trading risk。
4. 若某類樣本在 pnl_no_cost 仍有正期望，但在 pnl_cost_002 或 pnl_slip_025tick 下快速轉弱，代表它更接近 execution / cost-fragile 問題，而不一定是訊號本身失效。
5. Scenario proxy 表只用於 Future Work，不應直接把最高 PF 情境升級為主策略，除非它同時通過交易數、Train/Validation/Recent、成本後穩定性與市場邏輯檢查。
6. 下一步應從本檔案的 Tag_Summary、Tag_x_Session、Scenario_Proxy、Worst_Loss_Casebook 四張表中挑出 1~2 個最穩定的研究方向，放入作品集 Known Limitations / Future Work。
"""
conclusion = pd.DataFrame({"conclusion": [conclusion_text]})

readme = pd.DataFrame({
    "sheet": [
        "Parsed_Events", "Overall", "By_Split", "By_Session", "By_Exit", "Tag_Summary",
        "Tag_x_Split", "Tag_x_Session", "Tag_x_Exit", "Scenario_Thresholds",
        "Scenario_Proxy", "Scenario_By_Split", "Worst_Loss_Base", "Worst_Loss_Cost002", "Cost_Fragile_Cases", "Conclusion"
    ],
    "purpose": [
        "v2-B3 逐筆事件資料與高風險標籤",
        "四種成本情境下整體績效",
        "Train/Validation/Recent 分層",
        "時段分層，檢查午盤尾段是否成本後死亡",
        "出場原因分層，檢查 EXIT_TIME 是否為主要弱點",
        "高風險標籤總表，判斷主要風險來源",
        "高風險標籤 × 資料切分，避免只在單一期間有效",
        "高風險標籤 × 時段，檢查高風險是否集中在午盤/早盤",
        "高風險標籤 × 出場原因，檢查 EXIT_TIME 疊加效應",
        "proxy scenario 使用的分位數門檻",
        "Future Work proxy 情境，不是主策略定案",
        "各 proxy 情境的資料切分檢查",
        "未扣成本最差 30 筆案例",
        "成本 0.02 後最差 30 筆案例",
        "原始不虧但成本後死亡的薄利樣本",
        "研究結論與下一步"
    ]
})

# 顯示幾張核心表
print("\n=== Overall ===")
display(add_pretty_round(overall))
print("\n=== Tag Summary ===")
display(add_pretty_round(tag_perf.sort_values(["pnl_col", "net_pnl"])))
print("\n=== Scenario Proxy ===")
display(add_pretty_round(scenario_perf.sort_values(["pnl_col", "profit_factor"], ascending=[True, False])))

# 輸出 Excel
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    readme.to_excel(writer, sheet_name="README", index=False)
    review.to_excel(writer, sheet_name="Parsed_Events", index=False)
    add_pretty_round(overall).to_excel(writer, sheet_name="Overall", index=False)
    add_pretty_round(by_split).to_excel(writer, sheet_name="By_Split", index=False)
    add_pretty_round(by_session).to_excel(writer, sheet_name="By_Session", index=False)
    add_pretty_round(by_exit).to_excel(writer, sheet_name="By_Exit", index=False)
    add_pretty_round(by_signal_count).to_excel(writer, sheet_name="By_Signal_Count", index=False)
    add_pretty_round(by_rv).to_excel(writer, sheet_name="By_High_RV", index=False)
    add_pretty_round(tag_perf).to_excel(writer, sheet_name="Tag_Summary", index=False)
    add_pretty_round(tag_x_split).to_excel(writer, sheet_name="Tag_x_Split", index=False)
    add_pretty_round(tag_x_session).to_excel(writer, sheet_name="Tag_x_Session", index=False)
    add_pretty_round(tag_x_exit).to_excel(writer, sheet_name="Tag_x_Exit", index=False)
    add_pretty_round(session_x_exit).to_excel(writer, sheet_name="Session_x_Exit", index=False)
    add_pretty_round(repeated_x_session).to_excel(writer, sheet_name="Repeated_x_Session", index=False)
    add_pretty_round(highrv_x_exit).to_excel(writer, sheet_name="HighRV_x_Exit", index=False)
    thresholds.to_excel(writer, sheet_name="Scenario_Thresholds", index=False)
    add_pretty_round(scenario_perf).to_excel(writer, sheet_name="Scenario_Proxy", index=False)
    add_pretty_round(scenario_by_split).to_excel(writer, sheet_name="Scenario_By_Split", index=False)
    worst_loss_base.to_excel(writer, sheet_name="Worst_Loss_Base", index=False)
    worst_loss_cost002.to_excel(writer, sheet_name="Worst_Loss_Cost002", index=False)
    cost_fragile_cases.to_excel(writer, sheet_name="Cost_Fragile_Cases", index=False)
    conclusion.to_excel(writer, sheet_name="Conclusion", index=False)

print("\n✅ Exported:", OUTPUT_FILE)

if IN_COLAB:
    files.download(OUTPUT_FILE)
